# 5) Market coupling — Exercises

**Goals:** compute zonal PTDFs for any GSK, follow the money under nodal and zonal pricing, test whether a better GSK can make flow-based secure, draw a zone around the congestion, and find what a secure zonal market costs.

The setup cell holds the lecture's network, `PTDF`, `real_flows(dispatch)`, the nodal benchmark `NODAL` and `two_zone_market(zone, ntc, flow_based)` — the lecture's market for **any** split of the buses into zones "A" and "B", exchange E from A to B. `NORTH_SOUTH` is the lecture's split (A = South). Power in MW, prices in €/MWh, costs in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import xarray as xr

warnings.filterwarnings("ignore")
logging.getLogger("linopy").setLevel(logging.ERROR)

# ---- the lecture's network, plants, PTDFs and nodal benchmark ----
BUSES = ["Zagreb", "Rijeka", "Split", "Osijek", "Velebit"]
LINES = pd.DataFrame([("Zagreb", "Rijeka", 130), ("Rijeka", "Velebit", 110), ("Velebit", "Split", 160),
                      ("Zagreb", "Split", 320), ("Zagreb", "Osijek", 250), ("Osijek", "Split", 330)],
                     columns=["bus0", "bus1", "km"])
LINES.index = [f"{a}-{b}" for a, b in zip(LINES.bus0, LINES.bus1)]
X_KM, LIMIT = 0.32, 400.0
LOAD = pd.Series({"Zagreb": 900, "Rijeka": 300, "Split": 450, "Osijek": 250, "Velebit": 0})[BUSES]
GENS = pd.DataFrame([("wind Velebit", "Velebit", 600, 0.0), ("hydro Split", "Split", 700, 30.0),
                     ("gas Rijeka", "Rijeka", 600, 90.0), ("coal Osijek", "Osijek", 400, 80.0),
                     ("import Zagreb", "Zagreb", 800, 130.0)],
                    columns=["name", "bus", "p_max_MW", "cost_EUR_per_MWh"]).set_index("name")
NORTH_SOUTH = pd.Series({"Zagreb": "B", "Osijek": "B", "Rijeka": "A", "Velebit": "A", "Split": "A"})[BUSES]  # A = South


def ptdf(reference="Zagreb"):
    idx = {b: i for i, b in enumerate(BUSES)}
    b = 1 / (X_KM * LINES.km.to_numpy() / (400 ** 2 / 100))
    A = np.zeros((len(LINES), len(BUSES)))
    A[range(len(LINES)), LINES.bus0.map(idx)] = 1
    A[range(len(LINES)), LINES.bus1.map(idx)] = -1
    B = A.T @ np.diag(b) @ A
    keep = [i for i, name in enumerate(BUSES) if name != reference]
    B_inv = np.zeros_like(B)
    B_inv[np.ix_(keep, keep)] = np.linalg.inv(B[np.ix_(keep, keep)])
    return pd.DataFrame(np.diag(b) @ A @ B_inv, index=LINES.index, columns=BUSES)


PTDF = ptdf()
G = np.zeros((len(BUSES), len(GENS)))
for k, bus in enumerate(GENS.bus):
    G[BUSES.index(bus), k] = 1
GI = pd.Index(GENS.index, name="gen")
COST = xr.DataArray(GENS.cost_EUR_per_MWh.to_numpy(), coords=[GI])


def real_flows(dispatch):
    return pd.Series(PTDF.to_numpy() @ (G @ dispatch[GENS.index].to_numpy() - LOAD.to_numpy()), index=LINES.index)


def nodal():
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    m.add_constraints(p.sum() == LOAD.sum(), name="balance")
    M, c0 = PTDF.to_numpy() @ G, PTDF.to_numpy() @ LOAD.to_numpy()
    for l, line in enumerate(LINES.index):
        flow = (p * xr.DataArray(M[l], coords=[GI])).sum() - float(c0[l])
        m.add_constraints(flow <= LIMIT, name=f"upper {line}")
        m.add_constraints(flow >= -LIMIT, name=f"lower {line}")
    m.add_objective((p * COST).sum())
    m.solve(solver_name="highs", output_flag=False)
    lam = float(m.constraints["balance"].dual)
    mu = pd.Series({line: float(m.constraints[f"upper {line}"].dual) + float(m.constraints[f"lower {line}"].dual)
                    for line in LINES.index})
    return {"cost": float(m.objective.value), "dispatch": p.solution.to_pandas(), "lmp": lam + PTDF.T @ mu}


def two_zone_market(zone, ntc=None, flow_based=None):
    """The lecture's market for any split of the buses into zones "A" and "B" (Series bus -> zone).
    ntc limits the exchange E from A to B; flow_based is a list of (line, F_ref, k, E_ref).
    Returns None if the market has no solution."""
    gz = GENS.bus.map(zone)
    zl = LOAD.groupby(zone).sum()
    m = linopy.Model()
    p = m.add_variables(lower=0, upper=xr.DataArray(GENS.p_max_MW.to_numpy(), coords=[GI]), name="p")
    bound = 1e5 if ntc is None else ntc
    E = m.add_variables(lower=-bound, upper=bound, name="exchange")
    for z, sign in (("A", -1), ("B", 1)):
        gens = list(GENS.index[gz == z])
        produced = sum(p.sel(gen=g) for g in gens) if gens else 0
        m.add_constraints(produced + sign * E == zl.get(z, 0.0), name=f"balance {z}")
    for line, f_ref, k, e_ref in flow_based or []:
        m.add_constraints(k * E <= LIMIT - f_ref + k * e_ref, name=f"fb upper {line}")
        m.add_constraints(k * E >= -LIMIT - f_ref + k * e_ref, name=f"fb lower {line}")
    m.add_objective((p * COST).sum())
    status = m.solve(solver_name="highs", output_flag=False)
    if status[0] != "ok":
        return None
    dispatch = p.solution.to_pandas()
    flows = real_flows(dispatch)
    return {"cost": float(m.objective.value), "dispatch": dispatch, "exchange": float(E.solution), "flows": flows,
            "price": {z: abs(float(m.constraints[f"balance {z}"].dual)) for z in ("A", "B")},
            "secure": bool(flows.abs().max() <= LIMIT + 1e-6)}


NODAL = nodal()

### Warm-ups

**1. Zonal PTDFs for any GSK**

Write `exchange_ptdf(zone, gsk)`: the change in every line's flow per MW exported from zone A to zone B, when each zone's extra output is spread over its buses by the GSK (non-negative weights per bus, normalised within each zone). Check: if each zone's GSK sits on a single bus, the result is the difference of those two buses' PTDF columns; and with the lecture's capacity GSK you get the lecture's values (Zagreb–Rijeka −0.515, Rijeka–Velebit −0.200).

In [ ]:
def exchange_ptdf(zone: pd.Series, gsk: pd.Series) -> pd.Series:
    """Flow change on each line (MW) per MW exported from zone A to zone B, under the GSK."""
    ...  # YOUR CODE HERE

In [ ]:
single = exchange_ptdf(NORTH_SOUTH, pd.Series({"Velebit": 1.0, "Osijek": 1.0}))
assert np.allclose(single, PTDF["Velebit"] - PTDF["Osijek"])
capacity = GENS.groupby("bus").p_max_MW.sum()
k = exchange_ptdf(NORTH_SOUTH, capacity)
assert np.isclose(k["Zagreb-Rijeka"], -0.515, atol=1e-3) and np.isclose(k["Rijeka-Velebit"], -0.200, atol=1e-3)
k.round(3)

**2. Who pays?**

Compare the money flows of the nodal market and the one-zone market with redispatch: what consumers pay at market prices, what generators receive, the congestion rent (nodal) and the redispatch cost (zonal; cost-based, paid by the system operator and recovered from consumers through network tariffs). Write `who_pays()` returning a table with one column per design.

In [ ]:
def who_pays() -> pd.DataFrame:
    """Consumer payments, generator revenues, congestion rent and redispatch cost (EUR/h) of two designs."""
    ...  # YOUR CODE HERE

In [ ]:
money = who_pays()
assert np.isclose(money.loc["congestion rent (to network owner)", "nodal"], 86077.67, atol=0.1)
assert np.isclose(money.loc["redispatch cost (via tariffs)", "one zone + redispatch"], 10956.3, atol=0.1)
money.round(0)

### Core

**3. Can a better GSK make flow-based secure?**

Run the lecture's flow-based market (North/South, all lines monitored, base case = the one-zone dispatch) with three GSKs for the South: (a) by capacity (the lecture's), (b) only gas at Rijeka and hydro at Split — the plants the market actually adjusts, (c) only wind at Velebit. In the North keep capacity shares. For each, report market cost, exchange and the **real** flow on Rijeka–Velebit, or "no solution". Write `gsk_experiment()`.

In [ ]:
def gsk_experiment() -> pd.DataFrame:
    """Flow-based market (all lines) for three South GSKs."""
    ...  # YOUR CODE HERE

In [ ]:
gsks = gsk_experiment()
assert np.isclose(gsks.loc["(a) capacity", "market_cost"], 104586.0, atol=1)
assert (gsks.result != "secure").all(), "no GSK makes this flow-based market secure"
gsks

**4. A zone drawn around the congestion**

Draw a zone around Velebit alone (zone A = {Velebit}) and limit its export with an NTC of [600, 500, 485, 400] MW. For each, report market cost, wind output, the real flows on Zagreb–Rijeka and Rijeka–Velebit, and whether the dispatch is secure. Write `velebit_zone()`.

In [ ]:
def velebit_zone(ntcs=(600, 500, 485, 400)) -> pd.DataFrame:
    """NTC sweep for a zone around Velebit."""
    ...  # YOUR CODE HERE

In [ ]:
vz = velebit_zone()
assert not vz.secure.any()
assert np.isclose(vz.loc[485, "market_cost"], 81350.0, atol=1)
vz

**5. Interpretation**

Why does no GSK make the flow-based market secure — and why is (b), the GSK that describes the market's actual adjustments best, the one with no solution? Why does a zone around Velebit, with an NTC equal to the nodal wind output (485 MW), still overload a line?

*Your answer here.*

### Challenge

**6. What does a secure zonal market cost?**

For the North/South split and for the zone around Velebit, find the **largest** NTC (searching 0, 5, 10, … 800 MW) at which the zonal market's dispatch is secure, and its market cost. Compare with the nodal cost. Write `secure_ntc()`.

In [ ]:
def secure_ntc() -> pd.DataFrame:
    """Largest secure NTC (5 MW steps) and its market cost for two zone splits."""
    ...  # YOUR CODE HERE

In [ ]:
secure = secure_ntc()
assert (secure.above_nodal > 0).all()
secure.round(1)

**7. Interpretation**

How much more than the nodal optimum does a secure zonal market cost here, and why is the gap so large? What does that suggest about NTC-based zonal markets in meshed networks with several congested lines?

*Your answer here.*